# سبق 08 - ملٹی ایجنٹ ڈیزائن پیٹرن


## سیٹ اپ


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## کثیرالاجنبی نظام کیوں؟

حقیقی دنیا کے کام جیسے سفر کی منصوبہ بندی میں مختلف قسم کی مہارتیں شامل ہوتی ہیں — لاجسٹکس، مقامی معلومات، بجٹنگ اور بہت کچھ۔ ایک واحد ایجنٹ جو سب کچھ سنبھالنے کی کوشش کرتا ہے جلد ہی ناقابل انتظام ہو جاتا ہے۔

کثیرالاجنبی نظام اس مسئلے کو **ماہریت** کے ذریعے حل کرتے ہیں: ہر ایجنٹ ایک خاص مہارت کے علاقے پر توجہ دیتا ہے، جو ایک عمومی ایجنٹ کی نسبت زیادہ معیاری نتائج فراہم کرتا ہے۔ یہ نظام **پیمائش پذیری** کو بھی بہتر بناتے ہیں — آپ نئے ایجنٹس شامل کر سکتے ہیں (مثلاً فلائٹ اسپیشلسٹ، ریسٹورنٹ نقاد) بغیر موجودہ ورک فلو کو دوبارہ لکھے۔ ایجنٹس ایک منظم سلسلہ وار طریقہ سے ایک دوسرے کے ساتھ جڑتے ہیں، ایک سے دوسرے کو سیاق و سباق فراہم کرتے ہوئے۔


## ماہر ایجنٹس بنانا


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ایک ترتیب وار ورک فلو کی تخلیق

`WorkflowBuilder` آپ کو ایجنٹس کو ایک ہدایت شدہ گراف میں مربوط کرنے دیتا ہے۔ یہاں ہم ایک سادہ دو قدمی پائپ لائن بناتے ہیں: **TravelPlanner** روٹ پلان تیار کرتا ہے، پھر **TravelConcierge** اس کا جائزہ لیتا ہے اور اسے بہتر بناتا ہے۔


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ورک فلو میں مزید ایجنٹس کا اضافہ

کئی ایجنٹس کے نمونے کا سب سے بڑا فائدہ یہ ہے کہ اسے بڑھانا کتنا آسان ہے۔ نیچے ہم نے ایک **BudgetReviewer** ایجنٹ شامل کیا ہے جو منصوبے کو سفر کرنے والے کے بجٹ کے مقابلے میں چیک کرتا ہے، ان اشیاء کو نشان زد کرتا ہے جو اخراجات کو حد سے زیادہ کر سکتی ہیں، اور پیسے بچانے والی متبادلات تجویز کرتا ہے۔ ورک فلو اب تین ایجنٹس کو ترتیب سے چلاتا ہے:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ کیسے:

1. **ماہرانہ ایجنٹس بنائیں** — ہر ایک کا ایک مخصوص کردار ہو (منصوبہ بندی، کنسیئر، بجٹ جائزہ)۔
2. **ایجنٹس کو ایک ترتیب وار ورک فلو میں جوڑیں** `WorkflowBuilder` اور `add_edge` کا استعمال کرتے ہوئے۔
3. **کئی ایجنٹوں کی پائپ لائن سے آؤٹ پٹ کو اسٹریم کریں**، یہ معلوم کرتے ہوئے کہ کون سا ایجنٹ بول رہا ہے۔
4. **ایک ورک فلو کو بڑھائیں** نئے ایجنٹس کو چین میں شامل کرکے بغیر موجودہ ایجنٹس میں ترمیم کیے۔

کئی ایجنٹس کے ڈیزائن کا نمونہ ہر ایجنٹ کو سادہ رکھتا ہے جبکہ زیادہ جامع، اور زیادہ تفصیل سے جائزہ لیا ہوا نتیجہ پیدا کرتا ہے جو کہ کوئی بھی واحد ایجنٹ اکیلا حاصل نہیں کر سکتا۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**انتباہ**:  
اس دستاویز کو AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کیا گیا ہے۔ ہم درستگی کے لیے کوشاں ہیں، لیکن براہ کرم یاد رکھیں کہ خودکار ترجمہ میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنی مادری زبان میں معتبر ماخذ تصور کی جانی چاہیے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمہ کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کے لیے ہم ذمہ دار نہیں ہیں۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
